# 13.1 · 自编码器 / Autoencoder (AE)

> **课程定位 / Where this fits**
> 第 1 课，**Part 13 · 生成模型**。通往 VAE / 生成模型的起点。
> Lesson 1, **Part 13 · Generative Models**. The gateway to VAEs and generative modeling.
>
> 前面的模型多是**判别式**(给输入预测标签)。**生成模型**要做更难的事：**学会数据本身的分布, 从而生成全新的、像真的一样的数据**。**自编码器(autoencoder)** 是入门——它由**编码器**(把图压成一个低维"编码")和**解码器**(从编码重建图)组成, 中间的**瓶颈(bottleneck)** 逼模型学到数据的紧凑表示。AE 本身主要用于**降维/压缩/去噪/异常检测**, 但它直接孕育了能真正生成的 **VAE**(下一课)。本课从零搭 AE、在 MNIST 上训练、**可视化重建与隐空间**, 并揭示"为什么普通 AE 不能可靠地生成"。
> Earlier models were mostly **discriminative** (predict a label from input). **Generative models** do something harder: **learn the data distribution itself to generate brand-new, realistic data**. The **autoencoder** is the entry point — an **encoder** (compress an image into a low-dim "code") and a **decoder** (reconstruct from the code), with a **bottleneck** forcing a compact representation. AEs are mainly used for dimensionality reduction/compression/denoising/anomaly detection, but they directly birth the truly-generative **VAE** (next lesson). We build an AE from scratch, train on MNIST, **visualize reconstructions and the latent space**, and reveal "why a plain AE can't reliably generate."
>
> 💼 **实战/面试视角**："AE 结构/用途 / 瓶颈的作用 / AE 为什么不能直接当生成模型 / AE vs PCA" 是生成模型入门必问。
> 💼 **Practical/interview angle:** "AE structure/uses / role of the bottleneck / why AE isn't a generator / AE vs PCA" — generative entry must-knows.

> 📐 **符号约定 / Notation**
> - 编码 $z$ —— 输入被压成的低维向量(隐变量/latent) / the low-dim code (latent)
> - 重建 $\hat{x}$ —— 解码器还原出的输出 / the reconstruction
> - 瓶颈 —— 编码的维度(远小于输入) / the bottleneck dimension

> 💡 **面试相关 / Interview-relevant**
> - "自编码器的结构与训练目标"（出镜率 ★★★★）
> - "瓶颈/隐空间的作用"（★★★★）
> - "AE 为什么不能直接生成(隐空间有空洞)"（★★★★★）
> - "AE 和 PCA 的关系"（★★★，非线性降维）
> - "去噪自编码器/异常检测"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解生成 vs 判别, 以及自编码器的"压缩-重建"思想。
   Understand generative vs discriminative and the AE "compress-reconstruct" idea.
2. **从零搭 AE** 并在 MNIST 上训练, 可视化重建。
   Build an AE from scratch, train on MNIST, visualize reconstructions.
3. **可视化隐空间**, 看结构如何自发涌现。
   Visualize the latent space and watch structure emerge.
4. 揭示**普通 AE 不能可靠生成**的原因 → 引出 VAE。
   Reveal why a plain AE can't reliably generate → motivate VAE.

## 目录 / TOC
1. [生成模型与自编码器 ⭐](#1)
2. [搭建并训练 AE：重建 ⭐](#2)
3. [隐空间可视化 ⭐](#3)
4. [AE 能生成吗?局限与小结 ⭐](#4)


<a id="1"></a>
## 1. 生成模型与自编码器 ⭐ / Generative Models & Autoencoders

**判别模型** 学的是 $P(y \mid x)$(给图预测标签)；**生成模型** 学的是 $P(x)$(数据本身长什么样), 学会后就能**采样生成新数据**。生成更难——要理解"什么样的图像才像真实数字"。
**Discriminative** models learn $P(y \mid x)$ (label from image); **generative** models learn $P(x)$ (what the data itself looks like), enabling **sampling new data**. Harder — you must grasp "what makes an image look like a real digit."

**自编码器**的结构(它先学一个好的**表示**)：
The autoencoder's structure (it first learns a good **representation**):
```
输入 x  →[编码器 Encoder]→  编码 z (低维瓶颈)  →[解码器 Decoder]→  重建 x̂
input x →   encoder      →  code z (bottleneck) →   decoder      →  reconstruction x̂
```
训练目标极简：让**重建 $\hat{x}$ 尽量接近输入 $x$**(最小化重建误差, 如 MSE)。没有标签——这是**自监督**(标签就是输入自己)。
The objective is dead simple: make the **reconstruction $\hat{x}$ close to the input $x$** (minimize reconstruction error, e.g. MSE). No labels — **self-supervised** (the input is its own target).

**关键: 瓶颈**。如果编码维度和输入一样大, 模型可以"偷懒"原样复制。**瓶颈(编码维度远小于输入)** 逼它**丢掉冗余、只保留最本质的信息**——于是隐空间 $z$ 成为数据的紧凑表示。(AE 可看作 **PCA 的非线性推广**: 线性 AE ≈ PCA。)
**Key: the bottleneck.** If the code is as large as the input, the model can lazily copy. A **bottleneck (code ≪ input)** forces it to **drop redundancy, keep only the essence** — so the latent $z$ becomes a compact representation. (AE is a **nonlinear generalization of PCA**: a linear AE ≈ PCA.)


<a id="2"></a>
## 2. 搭建并训练 AE：重建 ⭐ / Build & Train: Reconstruction

在 **MNIST** 手写数字上训练。编码器把 784 维(28×28)的图压到一个低维编码, 解码器还原。看训练后**重建的数字**和原图多接近。
Train on **MNIST** digits. The encoder compresses the 784-dim (28×28) image to a low-dim code; the decoder restores it. We'll see how close the **reconstructions** are to the originals.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white"); torch.manual_seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.ToTensor()
train_full = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tfm)
test_full  = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=tfm)
train_loader = DataLoader(Subset(train_full, range(12000)), batch_size=128, shuffle=True)
print(f"MNIST: 训练用子集 12000 张 28×28 手写数字")

class Autoencoder(nn.Module):
    def __init__(self, latent=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784,256), nn.ReLU(),
                                     nn.Linear(256,64), nn.ReLU(), nn.Linear(64, latent))   # 压到 latent 维 / to latent
        self.decoder = nn.Sequential(nn.Linear(latent,64), nn.ReLU(), nn.Linear(64,256), nn.ReLU(),
                                     nn.Linear(256,784), nn.Sigmoid())                      # 还原到[0,1]的784维 / back to image
    def forward(self, x):
        z = self.encoder(x)                              # 编码 / encode
        return self.decoder(z).view(-1,1,28,28), z       # 解码重建 + 返回编码 / decode + return code

torch.manual_seed(0); ae = Autoencoder(latent=8); opt = torch.optim.Adam(ae.parameters(), 1e-3); mse = nn.MSELoss()
for epoch in range(12):
    for xb, _ in train_loader:                           # 无需标签(自监督) / no labels needed
        xhat, _ = ae(xb); loss = mse(xhat, xb)           # 重建误差 / reconstruction loss
        opt.zero_grad(); loss.backward(); opt.step()
print(f"AE 训练完成, 最终重建损失 = {loss.item():.4f}")

# 可视化: 原图 vs 重建 / visualize originals vs reconstructions
ae.eval()
imgs = torch.stack([test_full[i][0] for i in range(10)])
with torch.no_grad(): recon, _ = ae(imgs)
fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for i in range(10):
    axes[0,i].imshow(imgs[i,0], cmap="gray"); axes[0,i].axis("off")
    axes[1,i].imshow(recon[i,0], cmap="gray"); axes[1,i].axis("off")
axes[0,0].set_ylabel("原图", fontsize=10); axes[1,0].set_ylabel("重建", fontsize=10)
fig.suptitle("自编码器: 把数字压成8维编码再还原(重建可辨认 → 8维已抓住数字主要信息)")
plt.tight_layout(); plt.show()
print("重建图和原图很像 → 仅用8个数就编码了一个数字的主要信息(瓶颈逼模型学紧凑表示)")


<a id="3"></a>
## 3. 隐空间可视化 ⭐ / Latent Space Visualization

编码器把每张图映射成隐空间里的一个点。一个好的 AE 会让**隐空间有结构**——相似的数字(同一类、或写法相近)在隐空间里也靠近。我们把测试集编码后用 PCA 降到 2D, 按数字类别上色, 看是否成簇。
The encoder maps each image to a point in latent space. A good AE gives the **latent space structure** — similar digits land close together. We encode the test set, PCA to 2D, color by digit, and check for clusters.


In [ ]:
Xte = torch.stack([test_full[i][0] for i in range(2000)])
yte = np.array([test_full[i][1] for i in range(2000)])
with torch.no_grad(): _, Z = ae(Xte)                      # 编码到隐空间 / encode to latent
from sklearn.decomposition import PCA
Z2 = PCA(n_components=2).fit_transform(Z.numpy())         # 8维隐编码 → 2D 便于看 / PCA to 2D
fig, ax = plt.subplots(figsize=(7,6))
sc = ax.scatter(Z2[:,0], Z2[:,1], c=yte, cmap="tab10", s=8, alpha=0.7)
ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(sc, label="数字"); ax.set_title("AE 隐空间(PCA到2D): 同一数字自发聚成簇")
plt.tight_layout(); plt.show()
print("同类数字在隐空间聚在一起 → AE 学到了有意义的表示(没用标签, 自监督)")
print("用途: 降维/可视化(类似t-SNE) / 压缩 / 去噪(去噪AE) / 异常检测(重建误差大=异常)")


<a id="4"></a>
## 4. AE 能生成吗?局限与小结 ⭐ / Can AE Generate? Limits

既然解码器能"从编码生成图", 那**随便给解码器一个编码, 是不是就能生成新数字**? 试试: 从隐空间随机采点喂给解码器。
Since the decoder "generates an image from a code," can we **feed the decoder a random code to generate a new digit**? Let's try: sample random points in latent space and decode.


In [ ]:
# 尝试用 AE 生成: 从隐空间随机采样 → 解码 / try generating: sample random latent → decode
with torch.no_grad():
    z_range = Z.numpy()                                   # 训练编码的范围 / range of trained codes
    rand_z = torch.tensor(np.random.uniform(z_range.min(0), z_range.max(0), (10, 8)), dtype=torch.float32)
    gen = ae.decoder(rand_z).view(-1,1,28,28)
fig, axes = plt.subplots(1, 10, figsize=(12, 1.5))
for i in range(10): axes[i].imshow(gen[i,0], cmap="gray"); axes[i].axis("off")
fig.suptitle("用普通AE'生成': 随机采隐编码→解码 → 很多是模糊/不像数字的'四不像'")
plt.tight_layout(); plt.show()
print("问题: 普通AE的隐空间是'有空洞、不规则'的——训练只保证'真实数字编码处'能还原好,")
print("      但编码点之间的大片区域没人管 → 随机采样多半落在'空洞'里 → 解码出垃圾")
print("根因: AE 没有约束隐空间的分布(不知道该从哪儿采样)")
print("→ VAE(下一课)的关键改进: 强制隐空间服从一个已知分布(标准正态), 这样就能可靠采样生成!")


```
生成 vs 判别: 判别学P(y|x)预测标签; 生成学P(x)数据分布, 能采样生成新数据
自编码器: 编码器压成低维编码z(瓶颈) + 解码器重建x̂; 目标=最小化重建误差(MSE); 自监督(无标签)
瓶颈作用: 维度远小于输入→逼模型丢冗余留本质→学到紧凑表示; 线性AE≈PCA(AE是非线性推广)
隐空间: 同类样本自发聚簇; 用途=降维/压缩/去噪/异常检测(重建误差大=异常)
AE不能可靠生成: 隐空间有空洞/无已知分布→随机采样落到空洞→解码垃圾
→ VAE: 强制隐空间≈标准正态, 从而能可靠采样生成(下一课)
```

### 💡 面试速查 / Interview cheat-sheet
1. **AE结构**: 编码器→瓶颈编码→解码器; 目标=重建误差; 自监督。
   AE: encoder→bottleneck code→decoder; objective = reconstruction error; self-supervised.
2. **瓶颈**: 逼模型学紧凑本质表示(否则原样复制)。
   Bottleneck: forces a compact essential representation (else copy).
3. **AE vs PCA**: 非线性降维; 线性AE≈PCA。
   AE vs PCA: nonlinear dim-reduction; linear AE ≈ PCA.
4. **不能生成**: 隐空间有空洞、无已知分布, 随机采样→垃圾。
   Can't generate: latent has holes/no known distribution; random sampling → garbage.
5. **用途**: 降维/压缩/去噪AE/异常检测(重建误差)。
   Uses: dim-reduction/compression/denoising/anomaly detection (reconstruction error).

### 下一节 / Next
**13.2 变分自编码器(VAE)**——给 AE 的隐空间加上"必须服从标准正态分布"的约束, 它就变成了真正的生成模型: 从 N(0,1) 随便采一个点, 解码就能得到一个全新的、像样的数字。我们会讲清 ELBO、重参数化技巧, 并亲手生成数字。
**13.2 VAE** — constrain the AE's latent to follow a standard normal, and it becomes a true generative model: sample any point from N(0,1), decode, and get a new, plausible digit. We'll cover the ELBO, the reparameterization trick, and generate digits by hand.
